# Automated Assessment of Simulated Laparoscopic Surgical Performance  

This notebook demonstrates a cleaned implementation of the 3DCNN-based pipeline used for automated assessment of simulated laparoscopic surgical skill performance.  

It includes:  

- data loading  
- preprocessing  
- model definition  
- cross-validation training  
- test evaluation  
- visualisation of results  
- model saving  

📚 Citation

If you use this work, please cite the associated publications:


Power, D., Burke, C., Madden, M.G. et al. Automated assessment of simulated laparoscopic surgical skill performance using deep learning. Sci Rep 15, 13591 (2025). https://doi.org/10.1038/s41598-025-96336-5


D. Power and I. Ullah, "Automated Assessment of Simulated Laparoscopic Surgical Performance using 3DCNN," 2024 46th Annual International Conference of the IEEE Engineering in Medicine and Biology Society (EMBC), Orlando, FL, USA, 2024, pp. 1-4, https://doi.org/10.1109/EMBC53108.2024.10782160.




In [ ]:
import os
import cv2
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv3D,
    MaxPooling3D,
    BatchNormalization,
    Dropout,
    Flatten,
    Dense,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

In [ ]:
# Paths
TRAIN_DATASET_PATH = Path("data/train")
TEST_DATASET_PATH = Path("data/test")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Data / training settings
IMG_SIZE = (128, 128)
NUM_FRAMES = 60
NUM_FOLDS = 5
EPOCHS = 5
BATCH_SIZE = 1
NUM_CLASSES = 3   # change to 2 for binary setup if needed

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Dataset Structure  

The notebook expects the dataset to be organised in class-labelled folders, for example:  

data/  
├── train/  
│   ├── novice/  
│   ├── trainee/  
│   └── expert/  
└── test/  
    ├── novice/  
    ├── trainee/  
    └── expert/  

Each folder should contain video clips for the relevant class.  

In [ ]:
# video loading and preprocessing
def load_video(video_path, img_size=(128, 128), num_frames=60):
    frames = []
    cap = cv2.VideoCapture(str(video_path))

    while len(frames) < num_frames:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, img_size)
        frame = frame / 255.0
        frames.append(frame)

    cap.release()

    # Pad if shorter than required number of frames
    while len(frames) < num_frames:
        frames.append(np.zeros((img_size[1], img_size[0], 3)))

    return np.array(frames)


def load_dataset(dataset_path, img_size=(128, 128), num_frames=60):
    X = []
    y = []
    class_names = sorted([d.name for d in dataset_path.iterdir() if d.is_dir()])

    for label_idx, class_name in enumerate(class_names):
        class_dir = dataset_path / class_name
        for video_file in class_dir.glob("*"):
            if video_file.suffix.lower() in [".mp4", ".avi", ".mov", ".mkv"]:
                video_array = load_video(video_file, img_size=img_size, num_frames=num_frames)
                X.append(video_array)
                y.append(label_idx)

    return np.array(X), np.array(y), class_names

In [1]:
# build model
def build_3dcnn_model(input_shape, num_classes):
    model = Sequential()

    model.add(Conv3D(64, (3, 3, 3), activation="relu", padding="same", input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(MaxPooling3D(pool_size=(2, 2, 2)))
    model.add(Dropout(0.1))

    model.add(Conv3D(128, (3, 3, 3), activation="relu", padding="same"))
    model.add(BatchNormalization())
    model.add(MaxPooling3D(pool_size=(2, 2, 2)))
    model.add(Dropout(0.2))

    model.add(Conv3D(256, (3, 3, 3), activation="relu", padding="same"))
    model.add(BatchNormalization())
    model.add(MaxPooling3D(pool_size=(2, 2, 2)))
    model.add(Dropout(0.2))

    model.add(Conv3D(512, (3, 3, 3), activation="relu", padding="same"))
    model.add(BatchNormalization())
    model.add(MaxPooling3D(pool_size=(2, 2, 2)))
    model.add(Dropout(0.5))

    model.add(Flatten())
    model.add(Dense(1024, activation="relu"))
    model.add(Dropout(0.5))
    model.add(Dense(512, activation="relu"))

    if num_classes == 2:
        model.add(Dense(1, activation="sigmoid"))
        loss = "binary_crossentropy"
    else:
        model.add(Dense(num_classes, activation="softmax"))
        loss = "categorical_crossentropy"

    model.compile(
        optimizer=Adam(),
        loss=loss,
        metrics=["accuracy"]
    )

    return model

In [ ]:
# Load training data
X_train, y_train, class_names = load_dataset(
    TRAIN_DATASET_PATH,
    img_size=IMG_SIZE,
    num_frames=NUM_FRAMES
)

print("Training data shape:", X_train.shape)
print("Training labels shape:", y_train.shape)
print("Classes:", class_names)

In [ ]:
# Prepare labels
if NUM_CLASSES == 2:
    y_train_encoded = y_train
else:
    y_train_encoded = to_categorical(y_train, num_classes=NUM_CLASSES)

In [ ]:
# Cross-validation training loop
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)

histories = []
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), start=1):
    print(f"\\n--- Fold {fold}/{NUM_FOLDS} ---")

    X_tr, X_val = X_train[train_idx], X_train[val_idx]

    if NUM_CLASSES == 2:
        y_tr, y_val = y_train_encoded[train_idx], y_train_encoded[val_idx]
    else:
        y_tr, y_val = y_train_encoded[train_idx], y_train_encoded[val_idx]

    model = build_3dcnn_model(
        input_shape=(NUM_FRAMES, IMG_SIZE[1], IMG_SIZE[0], 3),
        num_classes=NUM_CLASSES
    )

    history = model.fit(
        X_tr,
        y_tr,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    histories.append(history)

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    fold_accuracies.append(val_acc)

    print(f"Fold {fold} validation accuracy: {val_acc:.4f}")

In [ ]:
# Cross-validation summary
print("Cross-validation accuracies:", fold_accuracies)
print("Mean CV accuracy:", np.mean(fold_accuracies))
print("Std CV accuracy:", np.std(fold_accuracies))

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 5))

for i, history in enumerate(histories, start=1):
    plt.plot(history.history["accuracy"], label=f"Fold {i} Train")
    plt.plot(history.history["val_accuracy"], linestyle="--", label=f"Fold {i} Val")

plt.title("Training and Validation Accuracy Across Folds")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
# Load test data
X_test, y_test, _ = load_dataset(
    TEST_DATASET_PATH,
    img_size=IMG_SIZE,
    num_frames=NUM_FRAMES
)

print("Test data shape:", X_test.shape)
print("Test labels shape:", y_test.shape)

In [ ]:
# Prepare test labels
if NUM_CLASSES == 2:
    y_test_encoded = y_test
else:
    y_test_encoded = to_categorical(y_test, num_classes=NUM_CLASSES)

In [ ]:
# Final model training on full training set
final_model = build_3dcnn_model(
    input_shape=(NUM_FRAMES, IMG_SIZE[1], IMG_SIZE[0], 3),
    num_classes=NUM_CLASSES
)

final_history = final_model.fit(
    X_train,
    y_train_encoded,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
# Test eval
test_loss, test_acc = final_model.evaluate(X_test, y_test_encoded, verbose=0)
print("Test accuracy:", test_acc)

In [ ]:
# generate predications and Confusion Matrix
y_pred_probs = final_model.predict(X_test)

if NUM_CLASSES == 2:
    y_pred = (y_pred_probs > 0.5).astype(int).flatten()
else:
    y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

In [ ]:
# classification report
report_text = classification_report(y_test, y_pred, target_names=class_names)
print(report_text)

In [ ]:
# plot Confusion Matrix
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# generate AUC/ROC
if NUM_CLASSES == 2:
    fpr, tpr, _ = roc_curve(y_test, y_pred_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.show()

In [ ]:
# save model
model_path = OUTPUT_DIR / "lap_skill_3dcnn_model.h5"
weights_path = OUTPUT_DIR / "lap_skill_3dcnn_weights.h5"

final_model.save(model_path)
final_model.save_weights(weights_path)

print("Model saved to:", model_path)
print("Weights saved to:", weights_path)

In [ ]:
# single example prediction demo
sample_idx = 0
sample_video = X_test[sample_idx:sample_idx+1]
sample_true = y_test[sample_idx]

sample_pred_probs = final_model.predict(sample_video)

if NUM_CLASSES == 2:
    sample_pred = int(sample_pred_probs.flatten()[0] > 0.5)
else:
    sample_pred = int(np.argmax(sample_pred_probs, axis=1)[0])

print("True label:", class_names[sample_true])
print("Predicted label:", class_names[sample_pred])